# Web Context Dataset Generation for Hateful Memes
Generates per-meme web-context embeddings (768-d, all-mpnet-base-v2) to be used as an external knowledge branch in the hate classifier.

**Fixes applied vs original notebook:**
- Label-agnostic queries (no label leakage into test splits)
- 1500-char clip instead of 512 (uses full fetched context)
- Mean-vector fallback instead of zero vector
- OCR key sanity check and coverage report
- Full checkpoint/resume support

In [4]:
# Add this at the top of your imports cell, before everything else
import warnings
import logging

# Suppress deprecation warnings (datetime.utcnow etc.)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Suppress jupyter/ipykernel warnings
logging.getLogger("jupyter_client").setLevel(logging.ERROR)
logging.getLogger("ipykernel").setLevel(logging.ERROR)

In [5]:
# Cell 1 — replace duckduckgo-search with ddgs
!pip install ddgs   # was duckduckgo-search>=6.1.0

In [6]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "duckduckgo-search>=6.1.0",
    "sentence-transformers>=2.6.0",
    "tqdm",
    "Pillow",
)
print("✅ Packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.1 MB/s eta 0:00:00
✅ Packages installed.


In [7]:
import os, json, pickle, time, random, warnings, re, glob, zipfile
from pathlib import Path
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import concurrent.futures
import torch

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_ROOT    = Path("/kaggle/input/datasets/williamberrios/hateful-memes/hateful_memes")
OCR_CHUNK_GLOB  = "/kaggle/input/datasets/krishna2215/ocr-chunk-zip/ocr_chunk_*.pkl"
OCR_ZIP_GLOB    = "/kaggle/input/datasets/krishna2215/ocr-chunk-zip/*.zip"

OUTPUT_DIR      = Path("/kaggle/working/web_context")
OCR_CACHE_DIR   = Path("/kaggle/working/ocr_cache")
LABEL_DIR       = Path("/kaggle/working/labels")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OCR_CACHE_DIR.mkdir(parents=True, exist_ok=True)
LABEL_DIR.mkdir(parents=True, exist_ok=True)

OCR_SAVE_PATH   = OCR_CACHE_DIR / "ocr_full_dataset.pkl"
RAW_PKL         = OUTPUT_DIR / "web_context_raw.pkl"
EMB_PKL         = OUTPUT_DIR / "web_context_embeddings.pkl"
CHECKPOINT_PKL  = OUTPUT_DIR / "_checkpoint_raw.pkl"

# ── Search config ─────────────────────────────────────────────────────────────
MAX_SNIPPETS      = 5       # snippets per meme
MAX_SNIPPET_CHARS = 300     # chars to keep per snippet
MAX_CONTEXT_CHARS = 1500    # FIXED: was 512 — use full fetched context before embedding
WORKERS           = 4       # parallel search threads
SLEEP_RANGE       = (0.4, 1.0)
CHECKPOINT_N      = 200     # save checkpoint every N memes
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Output dir : {OUTPUT_DIR}")
print(f"Device     : {DEVICE}")

Output dir : /kaggle/working/web_context
Device     : cuda


## Step 1 — Merge OCR chunks into a single lookup

In [8]:
OCR_CHUNK_GLOB = "/kaggle/input/datasets/arunjatothu/ocr-chunk/ocr_chunk_*.pkl"
OCR_SAVE_PATH  = OCR_CACHE_DIR / "ocr_full_dataset.pkl"

if OCR_SAVE_PATH.exists():
    print(f"📂 Consolidated OCR file already exists, loading...")
    with open(OCR_SAVE_PATH, "rb") as f:
        ocr_data = pickle.load(f)
    print(f"✅ Loaded {len(ocr_data)} entries ({OCR_SAVE_PATH.stat().st_size/1e6:.1f} MB)")
else:
    chunk_files = sorted(glob.glob(OCR_CHUNK_GLOB))
    if not chunk_files:
        raise FileNotFoundError(
            f"No OCR chunk files found at:\n  {OCR_CHUNK_GLOB}")

    print(f"🔨 Merging {len(chunk_files)} chunk files...")
    ocr_data = {}
    for chunk_path in tqdm(chunk_files, desc="Merging chunks"):
        with open(chunk_path, "rb") as f:
            chunk = pickle.load(f)
        for k, v in chunk.items():
            clean_key = str(k).replace("img/", "").replace("img\\", "").split("/")[-1].split("\\")[-1]
            ocr_data[clean_key] = v

    with open(OCR_SAVE_PATH, "wb") as f:
        pickle.dump(ocr_data, f)
    print(f"✅ Merged OCR saved → {OCR_SAVE_PATH} ({len(ocr_data)} entries)")


def _get_words(entry) -> List[str]:
    if isinstance(entry, dict):
        return entry.get("words", ["[UNK]"])
    if isinstance(entry, (list, tuple)) and len(entry) > 0:
        return entry[0] if isinstance(entry[0], list) else entry
    return ["[UNK]"]

ocr_lookup: Dict[str, List[str]] = {
    str(fname).split("/")[-1].split("\\")[-1]: _get_words(v)
    for fname, v in ocr_data.items()
}
print(f"✅ ocr_lookup ready — {len(ocr_lookup)} entries")

🔨 Merging 10 chunk files...


Merging chunks:   0%|          | 0/10 [00:00<?, ?it/s]

✅ Merged OCR saved → /kaggle/working/ocr_cache/ocr_full_dataset.pkl (12140 entries)
✅ ocr_lookup ready — 12140 entries


## Step 2 — Load and merge all dataset splits

In [9]:
SPLITS = {
    'train':       DATASET_ROOT / 'train.jsonl',
    'dev_seen':    DATASET_ROOT / 'dev_seen.jsonl',
    'dev_unseen':  DATASET_ROOT / 'dev_unseen.jsonl',
    'test_seen':   DATASET_ROOT / 'test_seen.jsonl',
    'test_unseen': DATASET_ROOT / 'test_unseen.jsonl',
}

dfs = []
for split_name, jsonl_path in SPLITS.items():
    if not jsonl_path.exists():
        print(f"⚠️  Missing: {jsonl_path}")
        continue
    rows = [json.loads(l) for l in open(jsonl_path)]
    df_s = pd.DataFrame(rows)
    df_s['split'] = split_name
    if 'label' not in df_s.columns:
        df_s['label'] = -1
    dfs.append(df_s)
    print(f"   {split_name:15s}: {len(df_s):5d} samples | hateful: {(df_s.label==1).sum()}")

full_df = pd.concat(dfs, ignore_index=True)

# Normalise image column
if 'img' in full_df.columns and 'image' not in full_df.columns:
    full_df['image'] = full_df['img']
full_df['img_id'] = full_df['image'].apply(
    lambda x: str(x).replace("img/", "").replace("img\\", "").split("/")[-1].split("\\")[-1]
)

csv_path = LABEL_DIR / 'hmc_info.csv'
full_df.to_csv(csv_path, index=False)
print(f"\n✅ Merged CSV saved → {csv_path} ({len(full_df)} rows)")

df = full_df.copy()
print(f"✅ df ready — {len(df)} rows, splits: {df['split'].value_counts().to_dict()}")

   train          :  8500 samples | hateful: 3019
   dev_seen       :   500 samples | hateful: 247
   dev_unseen     :   540 samples | hateful: 200
   test_seen      :  1000 samples | hateful: 490
   test_unseen    :  2000 samples | hateful: 750

✅ Merged CSV saved → /kaggle/working/labels/hmc_info.csv (12540 rows)
✅ df ready — 12540 rows, splits: {'train': 8500, 'test_unseen': 2000, 'test_seen': 1000, 'dev_unseen': 540, 'dev_seen': 500}


## Step 3 — OCR key sanity check

In [10]:
# FIXED: verify OCR keys actually match df img_ids before building queries
sample_img_id = df["img_id"].iloc[0]
if sample_img_id not in ocr_lookup:
    # Show what keys look like on each side to help debug
    print(f"❌ Key mismatch detected!")
    print(f"   df img_id sample    : {sample_img_id}")
    print(f"   ocr_lookup key sample: {list(ocr_lookup.keys())[0]}")
    raise AssertionError(
        f"'{sample_img_id}' not in ocr_lookup. "
        "Check prefix stripping in OCR merge step."
    )
else:
    print(f"✅ OCR key sanity check passed — sample: {sample_img_id}")

matched  = sum(1 for i in df["img_id"] if i in ocr_lookup)
missing  = len(df) - matched
print(f"OCR coverage: {matched}/{len(df)} ({100*matched/len(df):.1f}%)")
if missing > 0:
    print(f"⚠️  {missing} entries will fall back to '[UNK]' query")

✅ OCR key sanity check passed — sample: 42953.png
OCR coverage: 12540/12540 (100.0%)


## Step 4 — Query builder (label-agnostic)

In [11]:
def build_query(img_id: str, ocr_words: List[str]) -> str:
    """
    Build a focused, label-agnostic search query from OCR words.

    FIXED vs original: label parameter removed entirely.
    Original code used label=1 to prepend 'hateful meme meaning:' which
    leaks ground-truth labels into the retrieved context — the model could
    learn the prefix signal rather than meme semantics.
    All splits now use the same neutral prefix.
    """
    clean = [
        w.strip() for w in ocr_words
        if len(w.strip()) > 2 and not w.strip().isdigit()
    ]
    text_chunk = " ".join(clean[:12]) if clean else "meme"
    return f"meme context: {text_chunk}"


# Sanity check
print(build_query("01234.png", ["when", "you", "see", "the", "whole", "squad"]))
print(build_query("05678.png", ["dirty", "immigrant", "go", "back"]))
print(build_query("09999.png", []))  # fallback

meme context: when you see the whole squad
meme context: dirty immigrant back
meme context: meme


## Step 5 — Web snippet fetcher

In [12]:
# from duckduckgo_search import DDGS
from ddgs import DDGS

def fetch_web_snippets(
    query: str,
    max_results: int = MAX_SNIPPETS,
    max_chars: int = MAX_SNIPPET_CHARS,
    retries: int = 3,
) -> Tuple[List[str], str]:
    """
    Returns (list_of_snippets, concatenated_raw_text).
    Falls back to ([], '') on failure after all retries.
    """
    for attempt in range(retries):
        try:
            time.sleep(random.uniform(*SLEEP_RANGE))
            with DDGS() as ddgs:
                results = list(ddgs.text(query, max_results=max_results))
            snippets = []
            for r in results:
                body = r.get("body", "") or r.get("snippet", "")
                if body:
                    body = re.sub(r"\s+", " ", body).strip()[:max_chars]
                    snippets.append(body)
            raw_text = " [SEP] ".join(snippets)
            return snippets, raw_text
        except Exception as e:
            if attempt == retries - 1:
                return [], ""
            time.sleep(2 ** attempt)
    return [], ""


# Quick smoke test
snips, raw = fetch_web_snippets("meme context: stay home save lives covid")
print(f"Fetched {len(snips)} snippets, total chars: {len(raw)}")
if snips:
    print("First snippet:", snips[0][:120])

Fetched 5 snippets, total chars: 871
First snippet: An image of the full slogan "Stay alert, Control the virus, Save lives" was leaked by the Telegraph and led to edits and


## Step 6 — Build job list and resume from checkpoint

In [13]:
rows_jobs = []
for _, row in df.iterrows():
    img_id = str(row["img_id"])
    words  = ocr_lookup.get(img_id, ["[UNK]"])
    label  = int(row.get("label", -1))
    query  = build_query(img_id, words)   # label NOT passed — fixed
    rows_jobs.append({"img_id": img_id, "query": query, "label": label})

print(f"Total memes to process: {len(rows_jobs):,}")

# Resume from checkpoint
if CHECKPOINT_PKL.exists():
    with open(CHECKPOINT_PKL, "rb") as f:
        web_context_raw: Dict = pickle.load(f)
    print(f"Resumed from checkpoint: {len(web_context_raw):,} entries already done.")
else:
    web_context_raw: Dict = {}
    print("Starting fresh.")

todo = [r for r in rows_jobs if r["img_id"] not in web_context_raw]
print(f"Remaining: {len(todo):,}")

Total memes to process: 12,540
Starting fresh.
Remaining: 12,540


## Step 7 — Parallel web fetch with checkpointing

In [ ]:
def process_one(job: dict) -> Tuple[str, dict]:
    snippets, raw_text = fetch_web_snippets(job["query"])
    return job["img_id"], {
        "query":    job["query"],
        "snippets": snippets,
        "raw_text": raw_text if raw_text else "[NO_CONTEXT]",
        "label":    job["label"],
    }


completed = 0
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as executor:
    futures = {executor.submit(process_one, job): job for job in todo}
    pbar = tqdm(
        concurrent.futures.as_completed(futures),
        total=len(todo), desc="Fetching web context", ncols=100
    )
    for future in pbar:
        job = futures[future]
        try:
            img_id, result = future.result(timeout=30)
            web_context_raw[img_id] = result
            completed += 1
            if completed % CHECKPOINT_N == 0:
                with open(CHECKPOINT_PKL, "wb") as f:
                    pickle.dump(web_context_raw, f)
                pbar.set_postfix({"saved": completed, "total": len(web_context_raw)})
        except Exception as e:
            # Store empty result so we don't retry failed entries on resume
            web_context_raw[job["img_id"]] = {
                "query":    job["query"],
                "snippets": [],
                "raw_text": "[NO_CONTEXT]",
                "label":    job["label"],
            }

# Final save
with open(RAW_PKL, "wb") as f:
    pickle.dump(web_context_raw, f)
print(f"\n✅ Raw web context saved → {RAW_PKL} ({len(web_context_raw):,} entries)")

Fetching web context:   0%|                                               | 0/12540 [00:00<?, ?it/s]

## Step 8 — Encode with sentence-transformers (all-mpnet-base-v2)

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading sentence encoder: all-mpnet-base-v2 (768-d)")
encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
encoder.eval()
print("✅ Encoder loaded.")

In [ ]:
img_ids   = list(web_context_raw.keys())
raw_texts = [web_context_raw[i]["raw_text"] for i in img_ids]

# FIXED: was 512 chars — use MAX_CONTEXT_CHARS=1500 to pass full fetched context
# all-mpnet-base-v2 truncates internally at 384 tokens, so no OOM risk
raw_texts_clipped = [
    t[:MAX_CONTEXT_CHARS] if t else "[NO_CONTEXT]"
    for t in raw_texts
]

print(f"Encoding {len(img_ids):,} texts ...")
BATCH = 256
all_embeddings = []

with torch.no_grad():
    for start in tqdm(range(0, len(raw_texts_clipped), BATCH), desc="Embedding", ncols=80):
        batch_texts = raw_texts_clipped[start : start + BATCH]
        embs = encoder.encode(
            batch_texts,
            batch_size           = BATCH,
            show_progress_bar    = False,
            convert_to_numpy     = True,
            normalize_embeddings = True,   # unit-norm → cosine-friendly
        )
        all_embeddings.append(embs)

embeddings_matrix = np.vstack(all_embeddings)   # (N, 768)
print(f"Embeddings matrix shape: {embeddings_matrix.shape}")

# Build img_id → embedding lookup
web_context_embeddings: Dict[str, np.ndarray] = {
    img_id: embeddings_matrix[i]
    for i, img_id in enumerate(img_ids)
}

# FIXED: mean-vector fallback instead of zero vector
# Zero vectors have undefined cosine similarity and corrupt downstream layers.
# Mean of all valid embeddings is a neutral, well-normed fallback.
valid_mask   = np.array([web_context_raw[i]["raw_text"] != "[NO_CONTEXT]" for i in img_ids])
if valid_mask.sum() > 0:
    mean_vec = embeddings_matrix[valid_mask].mean(axis=0).astype(np.float32)
    mean_vec /= (np.linalg.norm(mean_vec) + 1e-8)   # re-normalise
else:
    mean_vec = np.zeros(768, dtype=np.float32)
    print("⚠️  All entries are NO_CONTEXT — check web fetch step")

all_img_ids = set(str(r["img_id"]) for r in rows_jobs)
missing_ids = [mid for mid in all_img_ids if mid not in web_context_embeddings]
for mid in missing_ids:
    web_context_embeddings[mid] = mean_vec.copy()

if missing_ids:
    print(f"⚠️  {len(missing_ids)} entries got mean-vector fallback")

with open(EMB_PKL, "wb") as f:
    pickle.dump(web_context_embeddings, f)

print(f"\n✅ Embeddings saved → {EMB_PKL} ({len(web_context_embeddings):,} entries)")
print(f"   Embedding dim : {next(iter(web_context_embeddings.values())).shape}")
print(f"   Dtype         : {next(iter(web_context_embeddings.values())).dtype}")

## Step 9 — Quality check

In [ ]:
# Coverage
no_ctx = sum(1 for v in web_context_raw.values() if v["raw_text"] == "[NO_CONTEXT]")
print(f"Entries with real context : {len(web_context_raw)-no_ctx:,} / {len(web_context_raw):,}")
print(f"No-context fallbacks      : {no_ctx:,}")

# Per-label stats
for lbl, name in [(0, "Not Hateful"), (1, "Hateful"), (-1, "Unlabelled")]:
    entries = [v for v in web_context_raw.values() if v["label"] == lbl]
    if entries:
        avg_snips = np.mean([len(v["snippets"]) for v in entries])
        avg_chars = np.mean([len(v["raw_text"])  for v in entries])
        print(f"  {name:15s}  n={len(entries):4d}  avg_snippets={avg_snips:.2f}  avg_chars={avg_chars:.0f}")

# Embedding norm check — should all be ~1.0 (unit norm)
norms = np.linalg.norm(np.stack(list(web_context_embeddings.values())), axis=1)
print(f"\nEmbedding L2 norm — mean: {norms.mean():.4f}  std: {norms.std():.4f}")
assert norms.mean() > 0.95, "❌ Norms are not close to 1.0 — check normalization"
print("✅ Norm check passed")

In [ ]:
# Show 3 sample entries
sample_ids = random.sample(img_ids, min(3, len(img_ids)))
for sid in sample_ids:
    entry = web_context_raw[sid]
    emb   = web_context_embeddings[sid]
    print(f"\n{'='*60}")
    print(f"  img_id   : {sid}")
    print(f"  label    : {entry['label']}")
    print(f"  query    : {entry['query']}")
    print(f"  #snips   : {len(entry['snippets'])}")
    print(f"  raw[:200]: {entry['raw_text'][:200]}")
    print(f"  emb[:6]  : {emb[:6]}  …  norm={np.linalg.norm(emb):.4f}")

print("\n✅ Dataset generation complete.")
print(f"   Raw context  : {RAW_PKL}")
print(f"   Embeddings   : {EMB_PKL}")
print("   Add both files as a Kaggle dataset for use in the classifier notebook.")